# Week 4 · Course 8 · Ensembles

**Sections**
1. Bagging — bootstrap aggregating, OOB error `(0:00 – 0:25)`
2. Random Forests — decorrelation, variable importance `(0:25 – 0:50)`
3. Boosting — sequential residual fitting, shrinkage `(0:50 – 1:15)`
4. BART — Bayesian Additive Regression Trees `(1:15 – 1:25)`
5. Summary — when to use which `(1:25 – 1:30)`

**Libraries:** `sklearn`, `numpy`, `pandas`, `matplotlib`, `seaborn`

## Setup

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingRegressor, BaggingClassifier,
    RandomForestRegressor, RandomForestClassifier,
# boosting fits trees sequentially, each correcting the previous residuals
    GradientBoostingRegressor, GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_squared_error, accuracy_score, roc_auc_score, ConfusionMatrixDisplay
)

from shared.data_utils import load_dataset

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
rng = np.random.default_rng(0)

In [ ]:
hitters  = load_dataset('hitters')
carseats = load_dataset('carseats')
boston   = load_dataset('boston')
kx_train = load_dataset('khan-xtrain')
ky_train = load_dataset('khan-ytrain')
kx_test  = load_dataset('khan-xtest')
ky_test  = load_dataset('khan-ytest')

# Heart disease — same as course-07
try:
    heart = pd.read_csv('https://www.statlearning.com/s/Heart.csv', index_col=0)
    heart = heart.dropna()
except Exception:
    rng2 = np.random.default_rng(42)
    n = 303
    age      = rng2.integers(29, 77, n)
    chol     = rng2.integers(130, 565, n).astype(float)
    thalach  = rng2.integers(71, 202, n).astype(float)
    sex      = rng2.choice(['Male', 'Female'], n)
    cp       = rng2.choice(['typical', 'atypical', 'non-anginal', 'asymptomatic'], n)
    thal     = rng2.choice(['normal', 'fixed', 'reversible'], n)
    logit    = (-6 + 0.05*age - 0.003*chol + 0.02*thalach
                + (sex == 'Male').astype(float)
                + (cp == 'asymptomatic').astype(float) * 2)
    prob     = 1 / (1 + np.exp(-logit))
    ahd      = rng2.binomial(1, prob).astype(str)
    ahd[ahd == '1'] = 'Yes'; ahd[ahd == '0'] = 'No'
    heart = pd.DataFrame({
        'Age': age, 'Sex': sex, 'ChestPain': cp, 'RestBP': rng2.integers(94, 200, n),
        'Chol': chol, 'Fbs': rng2.integers(0, 2, n), 'RestECG': rng2.integers(0, 3, n),
        'MaxHR': thalach, 'ExAng': rng2.choice([0, 1], n),
        'Oldpeak': np.clip(rng2.normal(1.0, 1.2, n), 0, 6.2).round(1),
        'Slope': rng2.integers(1, 4, n), 'Ca': rng2.integers(0, 4, n),
        'Thal': thal, 'AHD': ahd
    })

# Prepare Heart
h_heart = heart.copy()
cat_heart = [c for c in h_heart.select_dtypes('object').columns if c != 'AHD']
h_heart = pd.get_dummies(h_heart, columns=cat_heart, drop_first=True, dtype=float)
h_heart['AHD'] = (h_heart['AHD'] == 'Yes').astype(int)
X_hrt = h_heart.drop(columns=['AHD']); y_hrt = h_heart['AHD']
Xtr_hrt, Xte_hrt, ytr_hrt, yte_hrt = train_test_split(
    X_hrt, y_hrt, test_size=0.25, random_state=0, stratify=y_hrt)

# Prepare Hitters
h = hitters.dropna(subset=['Salary']).copy()
h['LogSalary'] = np.log(h['Salary'])
h_enc = pd.get_dummies(h.drop(columns=['Salary']),
                        columns=['League', 'Division', 'NewLeague'],
                        drop_first=True, dtype=float)
X_hit = h_enc.drop(columns=['LogSalary']); y_hit = h_enc['LogSalary']
Xtr_hit, Xte_hit, ytr_hit, yte_hit = train_test_split(
    X_hit, y_hit, test_size=0.25, random_state=0)

# Prepare Boston
Xb = boston.drop(columns=['medv']); yb = boston['medv']
Xtr_b, Xte_b, ytr_b, yte_b = train_test_split(Xb, yb, test_size=0.25, random_state=0)

print('All datasets ready.')
print(f'  Heart: {X_hrt.shape}  Hitters: {X_hit.shape}  Boston: {Xb.shape}')
print(f'  Khan: train {kx_train.shape}, test {kx_test.shape}')

---
## 1. Bagging (Bootstrap Aggregating)

**The core idea:** a single tree has low bias but high variance — its predictions change drastically if the training data changes slightly.  
Averaging many trees reduces variance without increasing bias.

### 1.1 The algorithm

1. Draw $B$ bootstrap samples from the training set (sample $n$ rows **with replacement**).
2. Fit a **deep, unpruned** decision tree to each bootstrap sample.
3. For **regression**: average the $B$ predictions.  
   For **classification**: majority vote (or average the class probabilities).

**Why it works:**  
Each tree is unbiased (deep trees fit well). Their predictions are correlated, but not perfectly. Averaging reduces variance by a factor approximately $\rho + (1-\rho)/B$ where $\rho$ is the pairwise tree correlation.  
As $B \to \infty$, bagging approaches the minimum achievable variance for trees trained on bootstrap samples from this distribution.
Bagging reduces variance without increasing bias — each tree is high-variance, but averaging cancels the noise. The bias of the average equals the bias of a single tree, while variance drops by 1/B for uncorrelated trees.


In [ ]:
# Manual bagging from scratch: see MSE converge as B increases
B_values = [1, 2, 5, 10, 25, 50, 100, 200, 300]
bagged_preds = np.zeros((len(B_values), len(Xte_b)))
mse_by_b = []

# Accumulate trees
all_trees = []
for b in range(max(B_values)):
    idx = rng.choice(len(Xtr_b), len(Xtr_b), replace=True)
    t = DecisionTreeRegressor(random_state=b)
    t.fit(Xtr_b.iloc[idx], ytr_b.iloc[idx])
    all_trees.append(t)

for i, B in enumerate(B_values):
    preds = np.mean([all_trees[b].predict(Xte_b) for b in range(B)], axis=0)
    mse_by_b.append(mean_squared_error(yte_b, preds))

# Single unpruned tree MSE
single_tree = DecisionTreeRegressor(random_state=0).fit(Xtr_b, ytr_b)
single_mse  = mean_squared_error(yte_b, single_tree.predict(Xte_b))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(B_values, mse_by_b, marker='o', label='Bagged trees')
ax.axhline(single_mse, color='C3', linestyle='--', label=f'Single tree ({single_mse:.2f})')
ax.set_xlabel('Number of bootstrap trees B')
ax.set_ylabel('Test MSE')
ax.set_title('Boston: MSE vs B (bagging converges quickly)')
ax.legend(); plt.show()
print(f'Single tree MSE:  {single_mse:.2f}')
print(f'Bagged 300 MSE:   {mse_by_b[-1]:.2f}')

### 1.2 OOB (Out-of-Bag) error — a free cross-validation

Each bootstrap sample uses approximately **63%** of the training rows (the rest are *out-of-bag*).  
For each training observation $i$, we average the predictions from all trees for which $i$ was **not** in the bootstrap sample.  
This gives an unbiased estimate of the test error — without any extra hold-out set!

> $\approx 1 - e^{-1} \approx 63.2\%$ of observations appear at least once in a bootstrap sample of the same size.

In [ ]:
# sklearn computes OOB error automatically
bag_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(),
# more trees = lower variance; diminishing returns beyond ~200
    n_estimators=300, oob_score=True, n_jobs=-1, random_state=0
).fit(Xtr_b, ytr_b)

oob_mse  = mean_squared_error(ytr_b, bag_reg.oob_prediction_)
test_mse = mean_squared_error(yte_b, bag_reg.predict(Xte_b))

print(f'OOB MSE  (on training set) = {oob_mse:.2f}')
print(f'Test MSE (on hold-out set) = {test_mse:.2f}')
print('OOB closely tracks test MSE — a free CV!')

---
## 2. Random Forests

Bagging still has a problem: if there is **one very strong predictor**, every tree will use it as the root split.  
The trees are **highly correlated** with each other, so averaging them gains little variance reduction.

**Random forests fix this by restricting each split to a random subset of $m$ features.**

- Classification default: $m = \lfloor\sqrt{p}\rfloor$
- Regression default: $m = \lfloor p/3 \rfloor$

By excluding the dominant feature from many splits, the trees are forced to explore other predictors → they become **decorrelated** → averaging gains much more.

### 2.1 Effect of m on test error
Bagging reduces variance without increasing bias — each tree is high-variance, but averaging cancels the noise. The bias of the average equals the bias of a single tree, while variance drops by 1/B for uncorrelated trees.


In [ ]:
# Vary max_features on Boston — reproduce ISLR Figure 8.10 style
p = Xtr_b.shape[1]
m_values = list(range(1, p + 1))
rf_mse = []
for m in m_values:
# more trees = lower variance; diminishing returns beyond ~200
    rf = RandomForestRegressor(n_estimators=200, max_features=m,
                                n_jobs=-1, random_state=0).fit(Xtr_b, ytr_b)
    rf_mse.append(mean_squared_error(yte_b, rf.predict(Xte_b)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(m_values, rf_mse, marker='.', label='Random Forest (200 trees)')
ax.axhline(mse_by_b[-1], color='C3', linestyle='--', label=f'Bagging (m=p={p}): {mse_by_b[-1]:.2f}')
ax.axvline(int(np.sqrt(p)), color='grey', linestyle=':', lw=1.5,
           label=f'√p = {int(np.sqrt(p))}')
ax.set_xlabel('m (features per split)')
ax.set_ylabel('Test MSE')
ax.set_title('Boston: test MSE vs m (random features per split)')
ax.legend(); plt.show()
best_m = m_values[int(np.argmin(rf_mse))]
print(f'Best m = {best_m}, MSE = {min(rf_mse):.2f}')
print(f'Bagging (m=p) MSE = {mse_by_b[-1]:.2f}')

### 2.2 Random forest on Heart (classification)

In [ ]:
# Single tree vs bagging vs RF on Heart
tree_hrt  = DecisionTreeClassifier(random_state=0).fit(Xtr_hrt, ytr_hrt)
# more trees = lower variance; diminishing returns beyond ~200
bag_hrt   = BaggingClassifier(n_estimators=300, n_jobs=-1, random_state=0).fit(Xtr_hrt, ytr_hrt)
# random feature subset at each split decorrelates trees — key to RF variance reduction
rf_hrt    = RandomForestClassifier(n_estimators=300, max_features='sqrt',
# OOB uses ~37% of samples not in each bootstrap as a free validation set
                                    oob_score=True, n_jobs=-1, random_state=0).fit(Xtr_hrt, ytr_hrt)

results = []
for name, m in [('Single tree', tree_hrt), ('Bagging 300', bag_hrt), ('Random Forest 300', rf_hrt)]:
    acc = m.score(Xte_hrt, yte_hrt)
    proba = m.predict_proba(Xte_hrt)[:, 1]
    auc = roc_auc_score(yte_hrt, proba)
    results.append({'Model': name, 'Test Accuracy': f'{acc:.4f}', 'Test AUC': f'{auc:.4f}'})

print(pd.DataFrame(results).to_string(index=False))
print(f'\nRF OOB error = {1 - rf_hrt.oob_score_:.4f}')

### 2.3 Variable importance

For each feature, average the **total impurity decrease** attributable to splits on that feature across all trees and all splits. Larger = more important.

> **Caveat:** impurity-based importance is biased toward high-cardinality features. Use `permutation_importance` for a more reliable alternative.
Impurity-based importance can inflate high-cardinality features with more candidate split points. Permutation importance (shuffle one feature, measure accuracy drop) is less biased and preferred when features differ greatly in cardinality.


In [ ]:
# Variable importance on Boston RF
# more trees = lower variance; diminishing returns beyond ~200
rf_b = RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=0).fit(Xtr_b, ytr_b)

# RF importance = mean impurity reduction across all trees
imp_gini = pd.Series(rf_b.feature_importances_, index=Xtr_b.columns).sort_values()

# Permutation importance (more reliable)
perm_imp = permutation_importance(rf_b, Xte_b, yte_b, n_repeats=20, random_state=0)
imp_perm = pd.Series(perm_imp.importances_mean, index=Xtr_b.columns).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
imp_gini.plot.barh(ax=axes[0]); axes[0].set_title('Gini importance (Boston RF)')
axes[0].set_xlabel('Mean impurity decrease')
imp_perm.plot.barh(ax=axes[1]); axes[1].set_title('Permutation importance (Boston RF)')
axes[1].set_xlabel('Mean MSE increase when permuted')
plt.tight_layout(); plt.show()
print('Top feature (Gini):       ', imp_gini.index[-1])
print('Top feature (Permutation):', imp_perm.index[-1])

In [ ]:
# Partial dependence plots for top 2 features
top2 = imp_gini.tail(2).index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
PartialDependenceDisplay.from_estimator(rf_b, Xte_b, top2, ax=axes)
plt.suptitle('Partial dependence: top-2 features (Boston RF)', y=1.02)
plt.tight_layout(); plt.show()

### 2.4 Gene expression data (Khan dataset)

The Khan dataset contains gene expression measurements for 4 types of small round blue cell tumours.  
There are 2,308 features (genes) but only 63 training observations — a **high-dimensional, low-sample** problem.  
Random forests handle this naturally by sampling features at every split.

In [ ]:
# Khan gene expression — RF classification
ky_tr = ky_train.iloc[:, 0].values
ky_te = ky_test.iloc[:, 0].values

rf_khan = RandomForestClassifier(
# more trees = lower variance; diminishing returns beyond ~200
    n_estimators=500, max_features='sqrt', n_jobs=-1, random_state=0
).fit(kx_train, ky_tr)

print(f'Khan gene expression — {kx_train.shape[1]} features, {kx_train.shape[0]} train, {kx_test.shape[0]} test')
print(f'RF train accuracy: {rf_khan.score(kx_train, ky_tr):.4f}')
print(f'RF test  accuracy: {rf_khan.score(kx_test, ky_te):.4f}')

# Top 10 most important genes
# RF importance = mean impurity reduction across all trees
imp_khan = pd.Series(rf_khan.feature_importances_, index=kx_train.columns)
top10 = imp_khan.nlargest(10)
fig, ax = plt.subplots(figsize=(7, 4))
top10.sort_values().plot.barh(ax=ax)
ax.set_title('Khan: top-10 important genes (RF)')
ax.set_xlabel('Gini importance')
plt.tight_layout(); plt.show()

---
## 3. Boosting

Unlike bagging/RF (which train trees **independently** in parallel), boosting trains trees **sequentially** — each tree corrects the errors of the ensemble so far.

### 3.1 The boosting algorithm (ISLR Algorithm 8.2)

1. Set $\hat{f}(x) = 0$ and residuals $r_i = y_i$ for all $i$.
2. For $b = 1, 2, \ldots, B$:
   a. Fit a tree $\hat{f}^b$ with $d$ splits to the current residuals $(x_i, r_i)$.
   b. Update: $\hat{f}(x) \leftarrow \hat{f}(x) + \lambda \hat{f}^b(x)$.
   c. Update residuals: $r_i \leftarrow r_i - \lambda \hat{f}^b(x_i)$.
3. Output: $\hat{f}(x) = \sum_{b=1}^B \lambda \hat{f}^b(x)$.

### Three tuning parameters

| Parameter | sklearn name | Effect |
|---|---|---|
| $B$ (# trees) | `n_estimators` | More trees → lower bias, risk of overfit |
| $\lambda$ (shrinkage) | `learning_rate` | Smaller → slower learning, need larger B, often better |
| $d$ (interaction depth) | `max_depth` | Depth-1 = stump (no interactions); depth-2 = one interaction |

**Key insight:** small $\lambda$ + large $B$ almost always beats large $\lambda$ + small $B$. Shrinkage prevents each tree from over-correcting.
Boosting reduces bias by sequentially fitting each new tree to the residuals of the ensemble so far — the opposite mechanism to bagging. Overfitting is controlled through the learning rate (shrinkage) and tree depth.


In [ ]:
# Boosting on Hitters salary: reproduce ISLR Figure 8.11
B_max = 500
results_boost = {}

for lr in [0.001, 0.01, 0.1]:
# boosting fits trees sequentially, each correcting the previous residuals
    gbm = GradientBoostingRegressor(
# more trees = lower variance; diminishing returns beyond ~200
        n_estimators=B_max, learning_rate=lr, max_depth=4,
        subsample=1.0, random_state=0
    ).fit(Xtr_hit, ytr_hit)
    # staged predictions: test MSE after each tree
    staged_mse = [
        mean_squared_error(yte_hit, pred)
        for pred in gbm.staged_predict(Xte_hit)
    ]
    results_boost[lr] = staged_mse

fig, ax = plt.subplots(figsize=(10, 5))
for lr, mse_curve in results_boost.items():
    ax.plot(range(1, B_max + 1), mse_curve, label=f'λ={lr}')
ax.set_xlabel('Number of trees B')
ax.set_ylabel('Test MSE')
ax.set_title('Boosting on Hitters log(Salary): test MSE vs B for different learning rates')
ax.legend(); plt.show()

for lr, mse_curve in results_boost.items():
    best_b = int(np.argmin(mse_curve)) + 1
    print(f'λ={lr:.3f}: best B={best_b}, min MSE={min(mse_curve):.4f}')

In [ ]:
# Best boosted model: variable importance
# boosting fits trees sequentially, each correcting the previous residuals
gbm_best = GradientBoostingRegressor(
# more trees = lower variance; diminishing returns beyond ~200
    n_estimators=500, learning_rate=0.01, max_depth=4, random_state=0
).fit(Xtr_hit, ytr_hit)

# RF importance = mean impurity reduction across all trees
imp_boost = pd.Series(gbm_best.feature_importances_, index=Xtr_hit.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
imp_boost.tail(10).plot.barh(ax=ax)
ax.set_title('Boosting: top-10 variable importance (Hitters log-salary)')
ax.set_xlabel('Relative importance')
plt.tight_layout(); plt.show()

In [ ]:
# Comparison: single tree / RF / boosting on Hitters salary
from sklearn.tree import DecisionTreeRegressor

models = {
    'Single tree (pruned)': None,
# more trees = lower variance; diminishing returns beyond ~200
    'Random Forest 300':    RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=0),
# boosting fits trees sequentially, each correcting the previous residuals
    'Boosting λ=0.01 B=500': GradientBoostingRegressor(n_estimators=500, learning_rate=0.01,
                                                        max_depth=4, random_state=0),
}

# Pruned tree
full_tree = DecisionTreeRegressor(random_state=0).fit(Xtr_hit, ytr_hit)
path = full_tree.cost_complexity_pruning_path(Xtr_hit, ytr_hit)
alphas = path.ccp_alphas[:-1]
cv_mses = [-cross_val_score(DecisionTreeRegressor(random_state=0, ccp_alpha=a),
                              Xtr_hit, ytr_hit, cv=5,
                              scoring='neg_mean_squared_error').mean()
           for a in alphas]
best_alpha = alphas[int(np.argmin(cv_mses))]
pruned_tree = DecisionTreeRegressor(random_state=0, ccp_alpha=best_alpha).fit(Xtr_hit, ytr_hit)
models['Single tree (pruned)'] = pruned_tree

rows = []
for name, m in models.items():
    if name != 'Single tree (pruned)':
        m.fit(Xtr_hit, ytr_hit)
    mse = mean_squared_error(yte_hit, m.predict(Xte_hit))
    rows.append({'Model': name, 'Test MSE': f'{mse:.4f}'})
print(pd.DataFrame(rows).to_string(index=False))

---
## 4. BART — Bayesian Additive Regression Trees

BART (Chipman, George, McCulloch 2010) is a Bayesian ensemble of trees.  
Like boosting, it sums many trees. But instead of fitting residuals greedily, it uses **MCMC** to explore the posterior distribution over all possible tree sums simultaneously.

### 4.1 The BART model

$$Y = \sum_{j=1}^{K} T_j(X) + \varepsilon, \quad \varepsilon \sim N(0, \sigma^2)$$

where each $T_j$ is a small decision tree (usually depth 1–2), and we place priors on the tree structure and leaf values.

### 4.2 The MCMC update (partial residual fitting)

At each MCMC iteration, for each tree $T_j$:
1. Compute **partial residuals**: $r_i = y_i - \sum_{k \neq j} T_k(x_i)$.
2. Perturb $T_j$: grow a leaf, prune a leaf, or change a split rule.
3. Accept/reject the perturbation based on how well the new $T_j$ fits the partial residuals.

After **burn-in** iterations (discarded), the posterior mean of the predictions is used as the final estimate.

### 4.3 Key differences from boosting

| | Boosting | BART |
|---|---|---|
| Training | Greedy forward steps | MCMC over full posterior |
| Uncertainty | None (point estimate) | Full posterior distribution |
| Overfitting | Needs early stopping / regularisation | Priors regularise naturally |
| Speed | Fast | Slow (MCMC) |
Boosting reduces bias by sequentially fitting each new tree to the residuals of the ensemble so far — the opposite mechanism to bagging. Overfitting is controlled through the learning rate (shrinkage) and tree depth.


In [ ]:
# BART conceptual simulation: sum of many shallow trees with partial residual updates
# This mimics the BART update rule without a full Bayesian implementation

def bart_like(X_train, y_train, X_test, K=50, n_iter=200, n_burn=50, max_depth=2, lr=0.1):
    """
    Simplified BART-like estimator: cycle through K trees, update each on partial residuals.
    Returns posterior-mean predictions from iterations after burn-in.
    """
    n_train = len(y_train)
    trees = [DecisionTreeRegressor(max_depth=max_depth, random_state=k) for k in range(K)]
    # Initialise: fit all trees on the full y
    for t in trees:
        idx = np.random.default_rng(id(t)).choice(n_train, n_train, replace=True)
        t.fit(X_train.iloc[idx], y_train.iloc[idx] / K)

    test_preds_accum = np.zeros(len(X_test))
    n_accum = 0

    for it in range(n_iter):
        for j, t in enumerate(trees):
            # Partial residual: remove this tree's contribution
            other_preds = sum(t2.predict(X_train) for k2, t2 in enumerate(trees) if k2 != j)
            partial_resid = y_train - other_preds
            # Perturb: refit on bootstrap of partial residuals
            idx = np.random.default_rng(it * K + j).choice(n_train, n_train, replace=True)
            t.fit(X_train.iloc[idx], partial_resid.iloc[idx])

        if it >= n_burn:
            full_pred = sum(t.predict(X_test) for t in trees)
            test_preds_accum += full_pred
            n_accum += 1

    return test_preds_accum / n_accum

print('Running BART-like estimator on Boston... (may take ~30s)')
bart_preds = bart_like(Xtr_b, ytr_b, Xte_b, K=50, n_iter=100, n_burn=30)
bart_mse = mean_squared_error(yte_b, bart_preds)

# Compare
rf_mse_b   = mean_squared_error(yte_b, rf_b.predict(Xte_b))
# more trees = lower variance; diminishing returns beyond ~200
gbm_b = GradientBoostingRegressor(n_estimators=500, learning_rate=0.01, max_depth=4, random_state=0)
gbm_b.fit(Xtr_b, ytr_b)
gbm_mse_b  = mean_squared_error(yte_b, gbm_b.predict(Xte_b))

print(f'\nBoston test MSE:')
print(f'  Single tree:   {mean_squared_error(yte_b, single_tree.predict(Xte_b)):.2f}')
print(f'  Random Forest: {rf_mse_b:.2f}')
print(f'  Gradient Boost:{gbm_mse_b:.2f}')
print(f'  BART-like:     {bart_mse:.2f}')

---
## 5. Summary — When to Use Which

| Method | Interpretable? | Accuracy | Speed | Notes |
|---|---|---|---|---|
| Single tree | ✅ High | Low–Med | Fast | Use for explanation only |
| Bagging | ❌ Low | Medium | Medium | Baseline ensemble |
| Random Forest | ❌ Low | High | Medium | Robust default for tabular |
| Gradient Boosting | ❌ Low | Very high | Slow | Often best on structured data |
| HistGBM (sklearn) | ❌ Low | Very high | Fast | Modern production default |
| BART | ❌ Low | High | Very slow | When uncertainty estimates needed |

**Practical guidance:**
- Start with `HistGradientBoostingClassifier` or `XGBoost` for production tabular problems.
- Use a single decision tree to *explain* a model to stakeholders.
- RF is a great second model — less tuning than GBM, very robust.
Bagging reduces variance without increasing bias — each tree is high-variance, but averaging cancels the noise. The bias of the average equals the bias of a single tree, while variance drops by 1/B for uncorrelated trees.


In [ ]:
# HistGradientBoosting — the modern fast default
t0 = time.perf_counter()
# boosting fits trees sequentially, each correcting the previous residuals
hgbm = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0)
hgbm.fit(Xtr_hrt, ytr_hrt)
t1 = time.perf_counter()
print(f'HistGBM — Heart test acc = {hgbm.score(Xte_hrt, yte_hrt):.4f} ')
print(f'          AUC = {roc_auc_score(yte_hrt, hgbm.predict_proba(Xte_hrt)[:,1]):.4f}')
print(f'          Time = {t1-t0:.2f}s')
print('Handles missing values natively (no imputation needed).')

---

# Exercises

Each exercise has a blank starter cell followed by a solution.

---
## Exercise 1

**Task.** Implement **bagging from scratch** on the **Hitters** log(Salary) prediction problem.  
For B in [1, 5, 10, 25, 50, 100, 200], compute test MSE.  
Plot MSE vs B and add the single pruned tree as a horizontal baseline.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

hitters = load_dataset('hitters')
h = hitters.dropna(subset=['Salary']).copy()
h['LogSalary'] = np.log(h['Salary'])
h_enc = pd.get_dummies(h.drop(columns=['Salary']),
                        columns=['League', 'Division', 'NewLeague'],
                        drop_first=True, dtype=float)
X = h_enc.drop(columns=['LogSalary']); y = h_enc['LogSalary']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
rng_ex = np.random.default_rng(1)

B_list = [1, 5, 10, 25, 50, 100, 200]
B_max = max(B_list)
all_t = []
for b in range(B_max):
    idx = rng_ex.choice(len(Xtr), len(Xtr), replace=True)
    t = DecisionTreeRegressor(random_state=b).fit(Xtr.iloc[idx], ytr.iloc[idx])
    all_t.append(t)

mse_bag = [mean_squared_error(yte, np.mean([t.predict(Xte) for t in all_t[:B]], axis=0))
           for B in B_list]

# Single pruned tree baseline
full = DecisionTreeRegressor(random_state=0).fit(Xtr, ytr)
path = full.cost_complexity_pruning_path(Xtr, ytr)
alphas = path.ccp_alphas[:-1]
cv_s = [-cross_val_score(DecisionTreeRegressor(random_state=0, ccp_alpha=a),
                          Xtr, ytr, cv=5, scoring='neg_mean_squared_error').mean()
        for a in alphas]
pruned = DecisionTreeRegressor(random_state=0,
                                ccp_alpha=alphas[int(np.argmin(cv_s))]).fit(Xtr, ytr)
single_mse = mean_squared_error(yte, pruned.predict(Xte))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(B_list, mse_bag, marker='o', label='Bagging')
ax.axhline(single_mse, color='C3', linestyle='--', label=f'Single pruned tree ({single_mse:.3f})')
ax.set_xlabel('B (bootstrap trees)'); ax.set_ylabel('Test MSE')
ax.set_title('Hitters: bagging MSE vs B'); ax.legend()
plt.show()
print(f'Single tree MSE: {single_mse:.4f}')
print(f'Bagging B=200:   {mse_bag[-1]:.4f}')

---
## Exercise 2

**Task.** Vary `max_features` from 1 to p on the **Hitters** log(Salary) regression problem (use 200 trees).  
Plot test MSE vs m.  
Mark the bagging baseline (m=p) and the RF default (m=p/3 for regression).

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

p = Xtr.shape[1]
m_vals = list(range(1, p + 1))
mse_m = []
for m in m_vals:
# more trees = lower variance; diminishing returns beyond ~200
    rf = RandomForestRegressor(n_estimators=200, max_features=m,
                                n_jobs=-1, random_state=0).fit(Xtr, ytr)
    mse_m.append(mean_squared_error(yte, rf.predict(Xte)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(m_vals, mse_m, marker='.')
ax.axvline(p, color='C3', linestyle='--', lw=1.5, label=f'Bagging (m=p={p})')
ax.axvline(max(1, p // 3), color='C2', linestyle=':', lw=1.5, label=f'RF default (m=p/3={p//3})')
ax.set_xlabel('m (features per split)'); ax.set_ylabel('Test MSE')
ax.set_title('Hitters: RF test MSE vs m'); ax.legend()
plt.show()
best = m_vals[int(np.argmin(mse_m))]
print(f'Best m = {best}, MSE = {min(mse_m):.4f}')

---
## Exercise 3

**Task.** Tune gradient boosting on **Hitters** log(Salary).  
Try all combinations of `learning_rate` in {0.01, 0.1} and `n_estimators` in {100, 300, 500}.  
Report test MSE for each combination.  
Which combination gives the lowest test MSE?

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
# boosting fits trees sequentially, each correcting the previous residuals
from sklearn.ensemble import GradientBoostingRegressor

rows = []
for lr in [0.01, 0.1]:
    for n in [100, 300, 500]:
        gbm = GradientBoostingRegressor(
# more trees = lower variance; diminishing returns beyond ~200
            n_estimators=n, learning_rate=lr, max_depth=4, random_state=0
        ).fit(Xtr, ytr)
        mse = mean_squared_error(yte, gbm.predict(Xte))
# learning_rate shrinks each tree's contribution — smaller = more regularisation
        rows.append({'learning_rate': lr, 'n_estimators': n, 'Test MSE': round(mse, 4)})

df_res = pd.DataFrame(rows)
print(df_res.to_string(index=False))
best = df_res.loc[df_res['Test MSE'].idxmin()]
print(f'\nBest: lr={best.learning_rate}, B={int(best.n_estimators)}, MSE={best["Test MSE"]}')

---
## Exercise 4

**Task.** On the **Heart** disease classification problem, compare: single decision tree, random forest (300 trees), and gradient boosting (300 trees, lr=0.05).  
For each model report: test accuracy, AUC, and training time.  
Plot the variable importance for the random forest.

In [ ]:
# your code here


### Exercise 4 — Solution

In [ ]:
import time
from sklearn.tree import DecisionTreeClassifier
# boosting fits trees sequentially, each correcting the previous residuals
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import pandas as pd

models_ex4 = [
    ('Single tree',     DecisionTreeClassifier(random_state=0)),
# more trees = lower variance; diminishing returns beyond ~200
    ('Random Forest',   RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0)),
# learning_rate shrinks each tree's contribution — smaller = more regularisation
    ('Gradient Boost',  GradientBoostingClassifier(n_estimators=300, learning_rate=0.05,
                                                    max_depth=3, random_state=0)),
]
rows = []
for name, m in models_ex4:
    t0 = time.perf_counter()
    m.fit(Xtr_hrt, ytr_hrt)
    t1 = time.perf_counter()
    acc = m.score(Xte_hrt, yte_hrt)
    auc = roc_auc_score(yte_hrt, m.predict_proba(Xte_hrt)[:, 1])
    rows.append({'Model': name, 'Accuracy': f'{acc:.4f}', 'AUC': f'{auc:.4f}',
                 'Train time (s)': f'{t1-t0:.2f}'})
print(pd.DataFrame(rows).to_string(index=False))

# Variable importance for RF
rf_ex4 = dict(models_ex4)['Random Forest']
# RF importance = mean impurity reduction across all trees
imp = pd.Series(rf_ex4.feature_importances_, index=X_hrt.columns).sort_values().tail(8)
fig, ax = plt.subplots(figsize=(6, 4))
imp.plot.barh(ax=ax)
ax.set_title('Heart RF: top-8 variable importances')
ax.set_xlabel('Gini importance')
plt.tight_layout(); plt.show()

---
## Exercise 5

**Task.** On the Heart dataset, fit `RandomForestClassifier(n_estimators=200, oob_score=True, random_state=0)`. Compare OOB error with 5-fold CV accuracy. Then plot OOB error vs `n_estimators` from 1 to 200 using `warm_start=True`. At roughly how many trees does OOB stabilise?


In [ ]:
# your code here


### Exercise 5 — Solution


In [ ]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from shared.data_utils import load_dataset

heart = load_dataset('heart').dropna()
heart_enc = pd.get_dummies(heart, columns=heart.select_dtypes('object').columns.tolist(), drop_first=True)
tcol = [c for c in heart_enc.columns if 'AHD' in c or 'target' in c.lower()][-1]
X_h = heart_enc.drop(columns=[tcol]).to_numpy(); y_h = heart_enc[tcol].to_numpy()

rf = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=0).fit(X_h, y_h)
print(f'OOB error:   {1 - rf.oob_score_:.4f}')
print(f'CV accuracy: {cross_val_score(rf, X_h, y_h, cv=5).mean():.4f}  (should be close)')

oob_errs = []
rf_w = RandomForestClassifier(n_estimators=1, warm_start=True, oob_score=True, random_state=0)
for n in range(1, 201):
    rf_w.n_estimators = n; rf_w.fit(X_h, y_h); oob_errs.append(1 - rf_w.oob_score_)

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(range(1,201), oob_errs)
ax.set(xlabel='n_estimators', ylabel='OOB error', title='OOB Convergence — Heart')
plt.tight_layout(); plt.show()
print('OOB typically stabilises around 50–100 trees.')


---
## Exercise 6

**Task.** On the Boston dataset (regression), fit `RandomForestRegressor` with `warm_start=True`, incrementing `n_estimators` from 1 to 300 in steps of 5. Record OOB R² at each step. Plot R² vs `n_estimators`. At roughly what point does adding more trees give <0.5% improvement?


In [ ]:
# your code here


### Exercise 6 — Solution


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from shared.data_utils import load_dataset

boston = load_dataset('boston')
X_b = boston.drop(columns=['medv']).to_numpy(); y_b = boston['medv'].to_numpy()

steps = list(range(1, 301, 5))
r2s = []
rf_b = RandomForestRegressor(n_estimators=1, warm_start=True, oob_score=True, random_state=0)
for n in steps:
    rf_b.n_estimators = n; rf_b.fit(X_b, y_b); r2s.append(rf_b.oob_score_)

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(steps, r2s)
ax.set(xlabel='n_estimators', ylabel='OOB R²', title='RF Convergence — Boston')
plt.tight_layout(); plt.show()
diffs = np.abs(np.diff(r2s))
idx = np.where(diffs < 0.005)[0]
if len(idx): print(f'R² stabilises ~n_estimators={steps[idx[0]]}')


---
## Exercise 7

**Task.** On the Heart dataset, compare `RandomForestClassifier(n_estimators=200)` vs `GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3)` using 5-fold CV AUC. Then sweep GBM `learning_rate` over `[0.001, 0.01, 0.05, 0.1, 0.3, 0.5]` and plot CV AUC vs learning rate.


In [ ]:
# your code here


### Exercise 7 — Solution


In [ ]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from shared.data_utils import load_dataset

heart = load_dataset('heart').dropna()
heart_enc = pd.get_dummies(heart, columns=heart.select_dtypes('object').columns.tolist(), drop_first=True)
tcol = [c for c in heart_enc.columns if 'AHD' in c or 'target' in c.lower()][-1]
X_h = heart_enc.drop(columns=[tcol]).to_numpy(); y_h = heart_enc[tcol].to_numpy()

rf  = RandomForestClassifier(n_estimators=200, random_state=0)
gbm = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=0)
print(f'RF  AUC: {cross_val_score(rf,  X_h, y_h, cv=5, scoring="roc_auc").mean():.4f}')
print(f'GBM AUC: {cross_val_score(gbm, X_h, y_h, cv=5, scoring="roc_auc").mean():.4f}')

lrs = [0.001, 0.01, 0.05, 0.1, 0.3, 0.5]
aucs = [cross_val_score(GradientBoostingClassifier(n_estimators=200,learning_rate=lr,max_depth=3,random_state=0), X_h, y_h, cv=5, scoring='roc_auc').mean() for lr in lrs]

fig, ax = plt.subplots(figsize=(7,4))
ax.semilogx(lrs, aucs, 'o-')
ax.set(xlabel='learning_rate (log)', ylabel='CV AUC', title='GBM Learning Rate — Heart')
plt.tight_layout(); plt.show()
print(f'Optimal learning_rate: {lrs[int(np.argmax(aucs))]}')
